# Introduction

In this notebook, data will be scraped using the Beautiful Soup library from a website storing car sale information. This data will then be serialized in a JSON format. Later, the data will be cleaned up and analysed, however for now all we need to do is write all the data to an appropriate file format.

In [3]:
# Import the beautiful soup library, url request library and JSON library
import bs4
import urllib.request
import json

# Parent link for the database. 
parent_link = "http://mlg.ucd.ie/modules/python/sources/cars/"
# This landing page contains links for different car brands
link = parent_link + "index.html" 

# Open the link and parse using the beautiful soup library
response = urllib.request.urlopen(link)
html = response.read().decode()
parser = bs4.BeautifulSoup(html, "html.parser")

## Data Model
The data will be stored in a large dictionary, where each brand is a different key. Each brand will contain a list of cars, each of which are a dictionary. Each car dictionary will contain the model anem and all the table information about that car scraped from the database.

## Parsing
### 1. Parse Landing Page


To find a list of all the brands, all the \<a href="brand_name"\> tags were found on the landing page. Using the find_all() method, the "a" tags could be found, and the href attribute information could be isolated.

### 2. First Brand Page

Each brand page contains multiple subpages, each of which contains different car data. A list of the different pages was created by find the \<nav\> tag at the bottom of the page. Each of the \<a\> tags in the nav were added to a list, along with the current page. Because the nav contains a "next" link, duplicates must be removed such that the second page is not accounted for twice.

### 3. Brand Subpages

Each of the subpages were then opened. Each car is represented by an \<li\> element. The name of the car is a \<h3\> element in this \<li\>, and all of the data is stored in a table. The individual elements of the table were then split by the ":" character to break it into the field and data. This data was added to the car dictionary. Each car dictionary then gets added to the car list.

Then the car list gets added to the brand dictionary. This brand dictionary is then dumped to a JSON file.

In [4]:
# Create a list of each car brand, and add the names of each car brand to the list
# This is done by finding all the links on the landing page
car_brand_list = []
for match in parser.find_all("a"):
    car_brand_list.append(match["href"]) # add all the links to a list
    
# For each brand, open the associated page and parse using beautiful soup
brand_json = {}
for brand in car_brand_list:
    brand_name = brand.split("-")[0]
    brand_link = parent_link + brand
    brand_response = urllib.request.urlopen(brand_link)
    brand_html = brand_response.read().decode()
    first_brand_parser = bs4.BeautifulSoup(brand_html, "html.parser")

    # Find list of all the pages you have to visit
    # Find the nav element at the bottom of the first page
    # Append all of the links in the nav to a list of all the pages
    page_list = [brand] # append the current page, because it wont appear in the links
    for nav in first_brand_parser.find_all("nav"):
        for a in nav.find_all("a"):
            page_list.append(a["href"])
    page_list = list(dict.fromkeys(page_list)) # Remove any duplicates, because "Next" makes page 2 appear twice

    # For each page, open the page and add the car info to a list storing their JSON information
    cars_json = []
    for page in page_list:
        # Parse the new page using beautiful soup
        page_link = parent_link + page
        page_response = urllib.request.urlopen(page_link)
        page_html = page_response.read().decode()
        page_parser = bs4.BeautifulSoup(page_html, "html.parser")
        
        # Each car in the website is associated with an li element
        for match in page_parser.find_all("li"):
            car_json_dict = {} # Create a new dictionary for the car
            for name in match.find_all("h3"):
                # Extract all the information for each car. The name is a h3 element
                car_name = name.get_text()
                car_json_dict["name"] = car_name
                
            # All the car data is stored in a table element. Add the data and the field to the car dictionary.
            for table in match.find_all("table"):
                for tr in table.find_all("tr"):
                    tr_tags = tr.find_all("td")
                    car_json_dict[tr_tags[0].get_text().strip(":")] = tr_tags[1].get_text()
            # Add the car dictionary to the list of cars
            cars_json.append(car_json_dict) 
    # Finally append the car dictionary to the brand list
    brand_json[brand_name] =  cars_json

# Open a data.json file and dump the json dictionary to it
with open('data.json', 'w') as f:
    json.dump(brand_json, f)
    
    